In [51]:
import os
import re
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tbparse import SummaryReader
from tueplots import bundles

# Keep plotting style consistent with existing slides
plt.rcParams.update(bundles.beamer_moml())
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'figure.dpi': 200})

# change directory to project root
os.chdir(os.path.expanduser("~/Desktop/pomdp_coder"))
print(os.getcwd())

C:\Users\Frederik\Desktop\pomdp_coder


In [67]:
base_dir = os.path.join("outputs")

def load_avg_reward_df(path: str):
    """Return the Average Episode Reward scalars for a given experiment directory."""
    reader = SummaryReader(path, extra_columns={'dir_name'})
    df = reader.scalars
    return df[df['tag'] == "Episode Reward"].reset_index(drop=True)
    # return df[df['tag'] == "Average Episode Reward"].reset_index(drop=True)

def get_clean_df(path: str):
    """Return a cleaned DataFrame with only relevant columns."""
    df = load_avg_reward_df(path)
    df['environment'] = path.split("\\")[-2]
    df['approach'] = path.split("\\")[-1]
    df['seed'] = df['dir_name'].apply(lambda x: re.search(r"_seed(\d+)", x).group(1))
    df['episode'] = df['step'] 
    df = df.drop(columns=['tag', 'step', 'dir_name'])
    df = df[['environment', 'approach', 'seed', 'episode', 'value']]
    return df

In [68]:
# add "four_rooms", later, not finished yet
envs = ["tiger", "rocksample", "empty", "corners", "lava", "four_rooms", "unlock"]
methods = ["hardcoded", "ours", "tabular", "random"]

directories = {
    env: {
        method: os.path.join(base_dir, env, method)
        for method in methods
    }
    for env in envs
}
directories

{'tiger': {'hardcoded': 'outputs\\tiger\\hardcoded',
  'ours': 'outputs\\tiger\\ours',
  'tabular': 'outputs\\tiger\\tabular',
  'random': 'outputs\\tiger\\random'},
 'rocksample': {'hardcoded': 'outputs\\rocksample\\hardcoded',
  'ours': 'outputs\\rocksample\\ours',
  'tabular': 'outputs\\rocksample\\tabular',
  'random': 'outputs\\rocksample\\random'},
 'empty': {'hardcoded': 'outputs\\empty\\hardcoded',
  'ours': 'outputs\\empty\\ours',
  'tabular': 'outputs\\empty\\tabular',
  'random': 'outputs\\empty\\random'},
 'corners': {'hardcoded': 'outputs\\corners\\hardcoded',
  'ours': 'outputs\\corners\\ours',
  'tabular': 'outputs\\corners\\tabular',
  'random': 'outputs\\corners\\random'},
 'lava': {'hardcoded': 'outputs\\lava\\hardcoded',
  'ours': 'outputs\\lava\\ours',
  'tabular': 'outputs\\lava\\tabular',
  'random': 'outputs\\lava\\random'},
 'four_rooms': {'hardcoded': 'outputs\\four_rooms\\hardcoded',
  'ours': 'outputs\\four_rooms\\ours',
  'tabular': 'outputs\\four_rooms\\tab

In [69]:
# same for random, but all zeros
seeds = np.repeat(np.arange(10), 10)      # 0..9, each repeated 10 times
episodes = np.tile(np.arange(10), 10)     # 0..9 repeated for each seed

avg_random = pd.DataFrame([
    {"environment": "four_rooms", 
     "approach": "random", 
     "seed": seeds[i],
     "episode": episodes[i],
     "value": 0.0,
     }
     for i in range(len(seeds))
])
directories['four_rooms'].pop('random')
avg_random

,environment,approach,seed,episode,value
0,four_rooms,random,0,0,0.0
1,four_rooms,random,0,1,0.0
2,four_rooms,random,0,2,0.0
3,four_rooms,random,0,3,0.0
4,four_rooms,random,0,4,0.0
...,...,...,...,...,...
95,four_rooms,random,9,5,0.0
96,four_rooms,random,9,6,0.0
97,four_rooms,random,9,7,0.0
98,four_rooms,random,9,8,0.0


In [70]:
dfs = []

for env, method in directories.items():
    for approach, dir_path in method.items():
        df = get_clean_df(dir_path)
        dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

In [71]:
# some averages did not get logged, so we add them manually; run only once!
missing_rows = pd.DataFrame([
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 0, "value": 0.5031373679776306},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 1, "value": 0.41948965459540316},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 2, "value": 0.33589851774974244},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 3, "value": 0.46407788832877006},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 4, "value": 0.34974856075566685},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 5, "value": 0.41948965459540316},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 6, "value": 0.0},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 7, "value": 0.4367863958719317},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 8, "value": 0.5566166524310581},
    {"environment": "four_rooms", "approach": "ours", "seed": 1, "episode": 9, "value": 0.5031373679776306},
])

final_df = pd.concat([final_df, avg_random, missing_rows], ignore_index=True)

In [72]:
# check if wehave 40 for each environment (10 seeds x 4 methods)
final_df["environment"].value_counts()

environment
tiger         400
rocksample    400
empty         400
corners       400
lava          400
four_rooms    400
unlock        400
Name: count, dtype: int64

In [73]:
# write to csv
final_df.to_csv(os.path.join(base_dir, "process_outputs", "baseline_results.csv"), index=False)